<a href="https://colab.research.google.com/github/Sricharan-rec/Gen-AI_Lab-AD23633/blob/main/gen_ai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# @title Exp_1A
import nltk
import random
from nltk.tokenize import word_tokenize
from nltk.tag import pos_tag, hmm
from nltk.probability import LidstoneProbDist

# ------------------ DOWNLOADS ------------------
nltk.download("punkt")
nltk.download("punkt_tab")   # FIX
nltk.download("averaged_perceptron_tagger_eng")

# ------------------ OWN DATASET ------------------
sentences = [
    "I love learning machine learning",
    "Machine learning is very interesting",
    "Data science helps students",
    "I enjoy working with data",
    "Artificial intelligence is powerful"
]

# ------------------ POS TAGGING ------------------
tagged_sentences = []

for sent in sentences:
    words = word_tokenize(sent.lower())
    tagged = pos_tag(words)
    tagged_sentences.append(tagged)

print("Tagged sentences:\n")
for s in tagged_sentences:
    print(s)

# ------------------ TRAIN HMM ------------------
trainer = hmm.HiddenMarkovModelTrainer()

hmm_model = trainer.train_supervised(
    tagged_sentences,
    estimator=lambda fd, bins: LidstoneProbDist(fd, 0.1, bins)
)

# =================================================
#               SENTENCE GENERATION
# =================================================

def generate_sentence_hmm(model, max_length=15):
    sampled = model.random_sample(random.Random(), max_length)
    sentence = [word for word, tag in sampled]
    return sentence

# ------------------ GENERATE ------------------
print("\nGenerated sentences:\n")

for _ in range(5):
    print(" ".join(generate_sentence_hmm(hmm_model, 12)))

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.


Tagged sentences:

[('i', 'NN'), ('love', 'VBP'), ('learning', 'VBG'), ('machine', 'NN'), ('learning', 'NN')]
[('machine', 'NN'), ('learning', 'NN'), ('is', 'VBZ'), ('very', 'RB'), ('interesting', 'JJ')]
[('data', 'NNS'), ('science', 'NN'), ('helps', 'VBZ'), ('students', 'NNS')]
[('i', 'NN'), ('enjoy', 'VBP'), ('working', 'VBG'), ('with', 'IN'), ('data', 'NNS')]
[('artificial', 'JJ'), ('intelligence', 'NN'), ('is', 'VBZ'), ('powerful', 'JJ')]

Generated sentences:

i powerful learning learning intelligence artificial machine interesting is very science working
data i machine learning machine data helps students enjoy working learning learning
i i i data interesting learning helps enjoy intelligence data very interesting
i is students very with students learning love learning working interesting with
learning is interesting students data science machine learning machine is intelligence science


In [6]:
# @title Exp_1B
import nltk
import random
from nltk.corpus import brown
from nltk.tag import hmm
from nltk.probability import LidstoneProbDist
from nltk.tokenize import word_tokenize

# ------------------ DOWNLOADS ------------------
nltk.download("brown")
nltk.download("punkt")
nltk.download("punkt_tab")

# ------------------ LOAD BROWN CORPUS ------------------
tagged_sentences = brown.tagged_sents()

print("Training sentences:", len(tagged_sentences))

# ------------------ TRAIN HMM WITH SMOOTHING ------------------
trainer = hmm.HiddenMarkovModelTrainer()

hmm_model = trainer.train_supervised(
    tagged_sentences,
    estimator=lambda fd, bins: LidstoneProbDist(fd, 0.1, bins)
)

# =============================================================
#          NEXT-WORD PREDICTION (USER INPUT CONTEXT)
# =============================================================

def predict_next_word_hmm(model, context_words):
    """
    Predict the next word using an HMM
    """

    # Infer POS tags using Viterbi
    tagged = model.tag(context_words)

    # Last POS tag
    last_state = tagged[-1][1]

    # Predict next POS
    next_state = model._transitions[last_state].generate()

    # Predict next word
    next_word = model._outputs[next_state].generate()

    return next_word


# ------------------ USER INPUT ------------------
user_input = input("\nEnter context sentence: ")

# Tokenize user input
context = word_tokenize(user_input.lower())

# Predict next word
predicted_word = predict_next_word_hmm(hmm_model, context)

print("\nContext:", context)
print("Predicted next word:", predicted_word)

[nltk_data] Downloading package brown to /root/nltk_data...
[nltk_data]   Package brown is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Training sentences: 57340

Enter context sentence: the government is

Context: ['the', 'government', 'is']
Predicted next word: to


In [ ]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 44.4 MB/s eta 0:00:00


In [ ]:
# @title Exp_2_A
import nltk
from nltk.corpus import brown
from gensim.models import Word2Vec

# Download corpus (run once)
nltk.download('brown')

# Load first 5000 sentences
sentences = brown.sents()[:5000]

# Preprocess (convert to lowercase)
processed_sentences = [[word.lower() for word in sent] for sent in sentences]

# -------------------------------
# Function to train model
# -------------------------------
def train_model(vector_size, window_size):
    model = Word2Vec(
        sentences=processed_sentences,
        vector_size=vector_size,
        window=window_size,
        sg=1,   # Skip-gram
        min_count=2,
        workers=4
    )
    return model

# -------------------------------
# Train models with different parameters
# -------------------------------
model1 = train_model(100, 2)   # small window
model2 = train_model(100, 5)   # medium window
model3 = train_model(200, 5)   # larger dimension

# -------------------------------
# Query words
# -------------------------------
query_words = ["money", "government", "student", "computer"]

# -------------------------------
# Function to print results
# -------------------------------
def show_similar_words(model, title):
    print(f"\n===== {title} =====")
    for word in query_words:
        if word in model.wv:
            print(f"\nTop 8 words similar to '{word}':")
            for similar_word, score in model.wv.most_similar(word,  topn=8):
                print(f"{similar_word} ({score:.4f})")
        else:
            print(f"\n'{word}' not in vocabulary")

# -------------------------------
# Display results
# -------------------------------
show_similar_words(model1, "Vector=100, Window=2")
show_similar_words(model2, "Vector=100, Window=5")
show_similar_words(model3, "Vector=200, Window=5")

[nltk_data] Downloading package brown to /root/nltk_data...
[nltk_data]   Unzipping corpora/brown.zip.



===== Vector=100, Window=2 =====

Top 8 words similar to 'money':
down (0.9979)
chicago (0.9977)
hole (0.9975)
office (0.9975)
sense (0.9974)
manager (0.9974)
court (0.9973)
gallery (0.9973)

Top 8 words similar to 'government':
back (0.9962)
pay (0.9961)
service (0.9960)
war (0.9955)
way (0.9952)
meeting (0.9951)
them (0.9951)
people (0.9951)

Top 8 words similar to 'student':
share (0.9985)
4 (0.9984)
near (0.9984)
commission (0.9984)
research (0.9983)
term (0.9983)
final (0.9983)
hotel (0.9983)

'computer' not in vocabulary

===== Vector=100, Window=5 =====

Top 8 words similar to 'money':
large (0.9963)
part (0.9956)
possible (0.9955)
matter (0.9952)
operation (0.9950)
belgians (0.9946)
business (0.9946)
war (0.9946)

Top 8 words similar to 'government':
increase (0.9947)
great (0.9945)
part (0.9931)
legislature (0.9929)
kind (0.9926)
early (0.9925)
money (0.9924)
education (0.9924)

Top 8 words similar to 'student':
near (0.9973)
garden (0.9973)
hole (0.9972)
graduate (0.9971)
13

In [ ]:
# @title Exp_2_B
from gensim.models import Word2Vec

# 1. Mini-corpus (10–15 sentences)
corpus = [
    "the cat sits on the mat",
    "the dog plays in the garden",
    "the boy reads a book",
    "the girl writes a letter",
    "students study in the library",
    "teachers teach in the school",
    "the sun shines brightly",
    "the moon glows at night",
    "birds fly in the sky",
    "fish swim in the water",
    "the computer processes data",
    "technology improves life",
    "money is important for living",
    "government makes policies",
    "people work for money"
]

# 2. Tokenization
sentences = [sentence.split() for sentence in corpus]

# 3. Train Skip-gram model
model = Word2Vec(
    sentences=sentences,
    vector_size=100,
    window=3,
    sg=1,
    min_count=1
)

# 4. Vocabulary
print("Vocabulary:", list(model.wv.index_to_key))

# 5. Query words
query_words = ["money", "computer", "school"]

# 6. Find similar words
for word in query_words:
    print(f"\nTop 5 similar words to '{word}':")
    for w, score in model.wv.most_similar(word, topn=5):
        print(f"{w} ({score:.4f})")

Vocabulary: ['the', 'in', 'for', 'money', 'a', 'work', 'people', 'policies', 'makes', 'government', 'living', 'important', 'is', 'life', 'improves', 'technology', 'data', 'processes', 'computer', 'water', 'swim', 'fish', 'sky', 'fly', 'birds', 'night', 'at', 'glows', 'moon', 'brightly', 'shines', 'sun', 'school', 'teach', 'teachers', 'library', 'study', 'students', 'letter', 'writes', 'girl', 'book', 'reads', 'boy', 'garden', 'plays', 'dog', 'mat', 'on', 'sits', 'cat']

Top 5 similar words to 'money':
improves (0.1781)
study (0.1639)
plays (0.1498)
work (0.1316)
sits (0.0968)

Top 5 similar words to 'computer':
in (0.1594)
school (0.1562)
for (0.1529)
boy (0.1500)
teachers (0.1447)

Top 5 similar words to 'school':
is (0.2379)
boy (0.1981)
important (0.1566)
computer (0.1562)
teach (0.1490)


In [ ]:
# @title Exp_2_C
import numpy as np
import gensim.downloader as api

# -------------------------------
# 1. Load pre-trained model
# -------------------------------
print("Loading model (this may take time)...")
model = api.load("fasttext-wiki-news-subwords-300")
print("Model loaded!")

# -------------------------------
# 2. Define WEAT word sets
# -------------------------------

# Gender sets
male = ["man", "male", "he", "him", "boy"]
female = ["woman", "female", "she", "her", "girl"]

# Attribute sets
career = ["career", "job", "work", "office", "business"]
family = ["home", "family", "children", "parents", "marriage"]

# -------------------------------
# 3. Cosine similarity helper
# -------------------------------
def cosine_sim(word, word_list):
    return np.mean([model.similarity(word, w) for w in word_list])

# -------------------------------
# 4. WEAT score calculation
# -------------------------------
def weat_score(X, Y, A, B):
    def s(w, A, B):
        return cosine_sim(w, A) - cosine_sim(w, B)

    return sum(s(x, A, B) for x in X) - sum(s(y, A, B) for y in Y)

# -------------------------------
# 5. Effect size calculation
# -------------------------------
def effect_size(X, Y, A, B):
    s_X = [cosine_sim(x, A) - cosine_sim(x, B) for x in X]
    s_Y = [cosine_sim(y, A) - cosine_sim(y, B) for y in Y]

    mean_diff = np.mean(s_X) - np.mean(s_Y)
    std_dev = np.std(s_X + s_Y)

    return mean_diff / std_dev

# -------------------------------
# 6. Run WEAT test
# -------------------------------
score = weat_score(male, female, career, family)
effect = effect_size(male, female, career, family)

# -------------------------------
# 7. Print results
# -------------------------------
print("\n--- WEAT Results ---")
print("WEAT Score:", score)
print("Effect Size:", effect)

# -------------------------------
# 8. Interpretation
# -------------------------------
if effect > 0:
    print("\nInterpretation: Bias detected (male → career, female → family)")
else:
    print("\nInterpretation: No strong bias detected")

Loading model (this may take time)...
[==================================================] 100.0% 958.5/958.4MB downloaded
Model loaded!

--- WEAT Results ---
WEAT Score: 0.1135658
Effect Size: 0.3725813

Interpretation: Bias detected (male → career, female → family)


In [1]:
# @title Exp_3
import tensorflow as tf
from tensorflow import keras

# Load & preprocess
(x_train, _), _ = keras.datasets.mnist.load_data()
x_train = (x_train.astype("float32") - 127.5) / 127.5
x_train = x_train.reshape(-1, 28, 28, 1)

# Params
batch_size, noise_dim, epochs = 64, 100, 10
dataset = tf.data.Dataset.from_tensor_slices(x_train).shuffle(60000).batch(batch_size)

# Generator
generator = keras.Sequential([
    keras.layers.Dense(128, activation='relu', input_shape=(noise_dim,)),
    keras.layers.Dense(784, activation='tanh'),
    keras.layers.Reshape((28, 28, 1))
])

# Discriminator
discriminator = keras.Sequential([
    keras.layers.Flatten(input_shape=(28,28,1)),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dense(1, activation='sigmoid')
])

# Loss & optimizers
loss_fn = keras.losses.BinaryCrossentropy()
g_opt = keras.optimizers.Adam(0.0002)
d_opt = keras.optimizers.Adam(0.0002)

# Training loop
for epoch in range(epochs):
    for real in dataset:
        bs = tf.shape(real)[0]
        noise = tf.random.normal((bs, noise_dim))

        # Train Discriminator
        with tf.GradientTape() as tape:
            fake = generator(noise)
            d_loss = loss_fn(tf.ones((bs,1)), discriminator(real))+
                     loss_fn(tf.zeros((bs,1)), discriminator(fake))
        grads = tape.gradient(d_loss, discriminator.trainable_variables)
        d_opt.apply_gradients(zip(grads, discriminator.trainable_variables))

        # Train Generator
        noise = tf.random.normal((bs, noise_dim))
        with tf.GradientTape() as tape:
            fake = generator(noise)
            g_loss = loss_fn(tf.ones((bs,1)), discriminator(fake))
        grads = tape.gradient(g_loss, generator.trainable_variables)
        g_opt.apply_gradients(zip(grads, generator.trainable_variables))

    print(f"Epoch {epoch+1}: D={d_loss:.3f}, G={g_loss:.3f}")

SyntaxError: invalid syntax (152844311.py, line 42)

In [ ]:
!pip install langchain_community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 42.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.4 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


In [ ]:
!pip install -U langchain langchain-community langchain-text-splitters langchainhuggingface faiss-cpu sentence-transformers

ERROR: Could not find a version that satisfies the requirement langchainhuggingface (from versions: none)
ERROR: No matching distribution found for langchainhuggingface


In [ ]:
!pip install langchain_huggingface

In [ ]:
!pip install -U langchain langchain-community langchain-text-splitters langchain-huggingface faiss-cpu sentence-transformers

  Using cached langchain-1.2.14-py3-none-any.whl.metadata (5.8 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.7/112.7 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 59.1 MB/s eta 0:00:00
  Attempting uninstall: langchain
    Found existing installation: langchain 1.2.12
    Uninstalling langchain-1.2.12:
      Successfully uninstalled langchain-1.2.12


In [ ]:
# @title Exp_5A
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# STEP 0: Create sample.txt (ADDED ONLY)
with open("sample.txt", "w", encoding="utf-8") as f:
    f.write("""Artificial Intelligence (AI) is a field of computer science that focuses on creating machines capable of performing tasks that typically require human intelligence.

Machine learning is a subset of AI that allows systems to learn from data and improve over time without being explicitly programmed.

Natural Language Processing (NLP) enables computers to understand, interpret, and generate human language.

Deep learning is a specialized area of machine learning that uses neural networks with many layers to analyze complex patterns in data.

AI is widely used in applications such as chatbots, recommendation systems, image recognition, and autonomous vehicles.
""")

# STEP 1: Load
loader = TextLoader("sample.txt", encoding="utf-8")
documents = loader.load()

# STEP 2: Split (UNCHANGED)
splitter = CharacterTextSplitter(
    separator="\n",
    chunk_size=500,
    chunk_overlap=50
)
split_docs = splitter.split_documents(documents)

# STEP 3: Embeddings
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# STEP 4: FAISS
vectorstore = FAISS.from_documents(
    documents=split_docs,
    embedding=embeddings
)

# STEP 5: Search
query = "What is this document about?"
results = vectorstore.similarity_search(query, k=3)

# STEP 6: Output
for i, doc in enumerate(results, start=1):
    print(f"\nResult {i}")
    print("-" * 50)
    print(doc.page_content)
    print("Metadata:", doc.metadata)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Result 1
--------------------------------------------------
Artificial Intelligence (AI) is a field of computer science that focuses on creating machines capable of performing tasks that typically require human intelligence.
Machine learning is a subset of AI that allows systems to learn from data and improve over time without being explicitly programmed.
Natural Language Processing (NLP) enables computers to understand, interpret, and generate human language.
Metadata: {'source': 'sample.txt'}

Result 2
--------------------------------------------------
Deep learning is a specialized area of machine learning that uses neural networks with many layers to analyze complex patterns in data.
AI is widely used in applications such as chatbots, recommendation systems, image recognition, and autonomous vehicles.
Metadata: {'source': 'sample.txt'}


In [2]:
# @title Exp_3
import tensorflow as tf
from tensorflow import keras

# Load & preprocess
(x_train, _), _ = keras.datasets.mnist.load_data()
x_train = (x_train.astype("float32") - 127.5) / 127.5
x_train = x_train.reshape(-1, 28, 28, 1)

# Params
batch_size, noise_dim, epochs = 64, 100, 10
dataset = tf.data.Dataset.from_tensor_slices(x_train).shuffle(60000).batch(batch_size)

# Generator
generator = keras.Sequential([
    keras.layers.Dense(128, activation='relu', input_shape=(noise_dim,)),
    keras.layers.Dense(784, activation='tanh'),
    keras.layers.Reshape((28, 28, 1))
])

# Discriminator
discriminator = keras.Sequential([
    keras.layers.Flatten(input_shape=(28,28,1)),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dense(1, activation='sigmoid')
])

# Loss & optimizers
loss_fn = keras.losses.BinaryCrossentropy()
g_opt = keras.optimizers.Adam(0.0002)
d_opt = keras.optimizers.Adam(0.0002)

# Training loop
for epoch in range(epochs):
    for real in dataset:
        bs = tf.shape(real)[0]
        noise = tf.random.normal((bs, noise_dim))

        # Train Discriminator
        with tf.GradientTape() as tape:
            fake = generator(noise, training=True)

            real_output = discriminator(real, training=True)
            fake_output = discriminator(fake, training=True)

            d_loss = (
                loss_fn(tf.ones((bs,1)), real_output) +
                loss_fn(tf.zeros((bs,1)), fake_output)
            )

        grads = tape.gradient(d_loss, discriminator.trainable_variables)
        d_opt.apply_gradients(zip(grads, discriminator.trainable_variables))

        # Train Generator
        noise = tf.random.normal((bs, noise_dim))
        with tf.GradientTape() as tape:
            fake = generator(noise, training=True)
            g_loss = loss_fn(tf.ones((bs,1)), discriminator(fake, training=True))

        grads = tape.gradient(g_loss, generator.trainable_variables)
        g_opt.apply_gradients(zip(grads, generator.trainable_variables))

    print(f"Epoch {epoch+1}: D={d_loss:.3f}, G={g_loss:.3f}")

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1: D=0.296, G=2.099
Epoch 2: D=0.433, G=1.686
Epoch 3: D=0.471, G=1.658
Epoch 4: D=0.426, G=1.929
Epoch 5: D=0.685, G=1.763
Epoch 6: D=0.428, G=2.019
Epoch 7: D=0.451, G=2.174
Epoch 8: D=0.353, G=2.515
Epoch 9: D=0.316, G=2.577
Epoch 10: D=0.453, G=2.435
